# 🛰️ Taller: Segmentación Shepherd **en tu navegador** (WebAssembly)

Este cuaderno corre **completo en tu navegador**: no instala nada en tu equipo, no usa la nube — el procesamiento ocurre en tu propia máquina, dentro de la pestaña.

Trabajaremos con un recorte real (512×512, 12 bandas) de la **geomediana Sentinel-2 2020 de Aguascalientes**, con el algoritmo de segmentación de **Shepherd et al. (2019)** — la misma implementación validada bit-a-bit contra `pyshepseg`.

Ejecuta cada celda con **Shift + Enter**. La primera tarda un poco (carga las bibliotecas científicas la primera vez).

In [ ]:
# 1) Bibliotecas (en el navegador, vía WebAssembly)
import time
import numpy as np
import rasterio
from rasterio import features
import matplotlib.pyplot as plt
import shepherd_pure

print("numpy", np.__version__, "| rasterio", rasterio.__version__, "(GDAL", rasterio.__gdal_version__ + ")")
print("✅ Stack geoespacial corriendo en WebAssembly")

In [ ]:
# 2) Leer el recorte de la geomediana (12 bandas Sentinel-2)
with rasterio.open("tile_ags_512.tif") as src:
    img = src.read().astype(np.float32)
    transform, crs, nodata = src.transform, src.crs, src.nodata
print(f"imagen: {img.shape[1]}x{img.shape[2]} px, {img.shape[0]} bandas, CRS {crs}")

# Composición RGB (bandas 4-3-2 de Sentinel-2 ≈ índices 3,2,1)
rgb = np.stack([img[3], img[2], img[1]], axis=-1)
rgb = np.clip((rgb - np.nanpercentile(rgb, 2)) / (np.nanpercentile(rgb, 98) - np.nanpercentile(rgb, 2)), 0, 1)
plt.figure(figsize=(6, 6)); plt.imshow(rgb); plt.title("Geomediana Sentinel-2 (RGB)"); plt.axis("off"); plt.show()

In [ ]:
# 3) Segmentación Shepherd (¡en tu navegador!)
t0 = time.time()
res = shepherd_pure.doShepherdSegmentation(
    img, numClusters=60, minSegmentSize=50,
    imgNullVal=nodata, fixedKMeansInit=True)
seg = res.segimg
print(f"{int(seg.max())} segmentos en {time.time()-t0:.1f} s")
print(f"píxeles sueltos eliminados: {res.singlePixelsEliminated:,}")
print(f"segmentos pequeños fusionados: {res.smallSegmentsEliminated:,}")

In [ ]:
# 4) Visualizar: fronteras de segmentos sobre la imagen
from scipy import ndimage
borders = (ndimage.maximum_filter(seg, size=2) != ndimage.minimum_filter(seg, size=2))
vis = rgb.copy(); vis[borders] = [1, 1, 0]
plt.figure(figsize=(7, 7)); plt.imshow(vis)
plt.title(f"Segmentación Shepherd — {int(seg.max())} segmentos"); plt.axis("off"); plt.show()

In [ ]:
# 5) Estadísticas espectrales por segmento — numpy puro (sin exactextract)
#    Los segmentos están alineados al píxel: bincount hace el trabajo exacto.
import pandas as pd
nseg = int(seg.max()) + 1
flat = seg.ravel()
total = np.bincount(flat, minlength=nseg)
tabla = {"segment_id": np.arange(1, nseg), "n_px": total[1:]}
for b in range(img.shape[0]):
    v = img[b].ravel()
    s = np.bincount(flat, weights=v, minlength=nseg)
    s2 = np.bincount(flat, weights=v * v, minlength=nseg)
    mean = np.where(total > 0, s / total, 0)
    tabla[f"b{b+1}Mean"] = mean[1:]
    tabla[f"b{b+1}StdDev"] = np.sqrt(np.maximum(np.where(total > 0, s2 / total, 0) - mean**2, 0))[1:]
df = pd.DataFrame(tabla)
print(f"tabla de features: {df.shape[0]} segmentos × {df.shape[1]-2} features espectrales")
df.head()

In [ ]:
# 6) Poligonizar y pintar la media espectral por segmento
import geopandas as gpd
geoms = ({"properties": {"segment_id": int(v)}, "geometry": g}
         for g, v in features.shapes(seg.astype(np.int32), transform=transform) if v != 0)
gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs).dissolve(by="segment_id", as_index=False)
gdf = gdf.merge(df[["segment_id", "b8Mean"]], on="segment_id")
ax = gdf.plot(column="b8Mean", cmap="RdYlGn", figsize=(7, 7), linewidth=0)
ax.set_title("Media de banda 8 (NIR) por segmento — objetos, no píxeles")
ax.set_axis_off(); plt.show()
print(f"{len(gdf)} polígonos georreferenciados — listos para clasificación")

---
### 🎓 Lo que acabas de hacer

Segmentaste una imagen satelital real con el algoritmo Shepherd, calculaste las features espectrales de cada objeto y los convertiste en polígonos georreferenciados — **todo dentro de tu navegador**, sin instalar nada. Con estas features, el siguiente paso del curso es entrenar un clasificador (`scikit-learn` también corre aquí).

**Curso de análisis de imágenes satelitales — Dr. Abel Coronado.**